# Datamine MIKSCELL Process Example

This notebook demonstrates how to configure and run the **`mikscell`** process wrapper in `dmstudio`.

## Process Description

## MIKSCELL

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

**MIKSCELL** is designed to take a model created by Multiple Indicator Kriging (MIK) and convert it into a model that is suitable for input to Studio NPVS or Studio NPVS+

In the MIK model each record includes a pair of fields, **PRABi** and **GRABi** , i=1,n, containing the PRoprtion ABove cut off and the GRade ABove cutoff for a set of n cutoff grades. In the output model the interval between successive cutoff grades is represented by an individual subcell.

The volume of each subcell represents the volume of material between the cutoffs. The dimensions of the subcell are set equal to the dimensions of the parent cell in two directions and in the third direction the size is calculated so that the volume is correct. Parameter AXIS defines the method used for determining the subcell size.

Parameter **MINVOL** allows the user to specify the minimum volume of an individual subcell. If a subcell volume is less than the minimum it will be combined with the subcell in the grade interval below. However note that if the total volume of all subcells is less than **MINVOL** the combined subcell will still be included in the output model.

The output model includes field **INTERVAL**. This is set to 1 for the subcell that represents the material lower than the first cutoff, 2 for material between the first and second cutoffs, and so on. If **MINVOL** has been selected and two or more subcells have been combined then the **INTERVAL** values will also be averaged so non integer values of **INTERVAL** will be created.

### Input Files:

* **modelin** (Block Model):
  Convert Multiple Indicator Kriging (MIK) output created by **[ESTIMA](<estima.md>)** or
  **[INDEST](<indest.md>)** to individual subcells for each grade range.
  Required=Yes

### Output Files:

* **modelout** (Input):
  Block Model
  Required=Yes

### Fields:

### Parameters:

* **minvol**:
  The minimum volume of a subcell. If the subcell is less than the minimum it will be
  combined with the subcell with the next lowest grade. If the lowest grade subcell is
  less than **MINVOL** it will be combined with the subcell in thegrade range above
  Range=
  Values=
  Default=0
  Required=No

* **tolernce**:
  This defines the smallest subcell that will be included in **MODELOUT** as a proportion
  of the parent cell size. The default of 0.0001 means that the subcell size along each
  axis cannot be less than 0.01% of the parent cell size.
  Range=
  Values=
  Default=0.0001
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('mikscell')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute mikscell
print("Running mikscell...")
dm_cmd.mikscell(
    modelin_i='t_mod1',  # required
    modelout_o='t_mikscell_out',  # required
    # minvol_p=0,  # optional
    # tolernce_p=0.0001,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("mikscell execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_mikscell_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")